# Pré-requis : Dataset, Tête de classification, Test split commun

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from constants import FEATURES_DIR
import os
import mlflow
import mlflow.pytorch

# Config
# ====
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ====
# Dataset
# ====
class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ====
# ResNet50 head (classification) on top of ResNet50 embeddings (avgpool)
# ====
class ResNet50Head(nn.Module): ## nn.Module => héritage réseau de neuronnes pytorch
    """
    Classification head qui prend en entrée des features extraites par ResNet50
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        return self.net(x)

# ====
# CRÉATION DU TEST MIXTE COMMUN (pour comparaison équitable),
# et DU SPLIT STRONG (pour entrainement supervisé)
# ====

# 1. Préparation des données Strong (100 images)
X_strong = np.vstack([
    np.load(FEATURES_DIR / "features_cancer_avgpool.npy"),
    np.load(FEATURES_DIR / "features_normal_avgpool.npy")
])
y_strong = np.array([1] * 50 + [0] * 50)

# 2. SPLIT STRICT : 80 pour train, 20 pour test (strong)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_strong,
    y_strong,
    test_size=20,
    stratify=y_strong,
    random_state=SEED
)

# 3. Préparation des données Weak (pour compléter le test ET pour le semi-supervisé)

# Charger TOUTES les weak
X_weak_all = np.load(FEATURES_DIR / "features_unlabeled_avgpool.npy")
y_weak_all = np.load(FEATURES_DIR / "weak_labels.npy")

# Séparer une fois pour toutes :
# - 180 images weak pour le TEST MIXTE (X_test_w, y_test_w)
# - le reste pour l'ENTRAINEMENT/VALIDATION SEMI-SUPERVISÉ (X_weak_rest, y_weak_rest)
X_weak_rest, X_test_w, y_weak_rest, y_test_w = train_test_split(
    X_weak_all,
    y_weak_all,
    test_size=180,
    stratify=y_weak_all,
    random_state=SEED
)

# 4. Création du Test Set Mixte (20 strong + 180 weak)
X_test_mixed = np.vstack([X_test_s, X_test_w])
y_test_mixed = np.concatenate([y_test_s, y_test_w])

# 5. Mélange du test set
idx = np.random.permutation(len(y_test_mixed))
X_test_mixed, y_test_mixed = X_test_mixed[idx], y_test_mixed[idx]

print(f"Entraînement supervisé : {len(X_train_s)} images strong")
print(f"Test mixte : {len(X_test_mixed)} images (dont 20 strong jamais vues par les modèles)")

# 6. Création du dataset pour l'approche SEMI-SUPERVISÉE (sur le reste des weak)
dataset = FeatureDataset(X_weak_rest, y_weak_rest)
print(f"Semi-supervisé : {len(dataset)} images weak disponibles (hors test mixte)")

Entraînement supervisé : 80 images strong
Test mixte : 200 images (dont 20 strong jamais vues par les modèles)
Semi-supervisé : 1226 images weak disponibles (hors test mixte)


# Approche Semi-supervisée

In [2]:
# ====
# Approche Semi-supervisée
# ====

# MLflow Config
MLFLOW_EXPERIMENT_NAME = "resnet50-weaklabels"

# ====
# # Load data
# ====
print("\nChargement des features.")
files = [f for f in os.listdir(FEATURES_DIR) if f.endswith(".npy")]
X = np.load(FEATURES_DIR / "features_unlabeled_avgpool.npy")
y = np.load(FEATURES_DIR / "weak_labels.npy")

print(f"\nSamples weak-labeled: {len(y)}")
print(f"X shape: {X.shape}")
print(f"Distribution: {np.bincount(y)}")

# ====
# # Train / Val split
# ====
train_size = int(0.8 * len(dataset)) # 80% des données pour l'entrainement
val_size = len(dataset) - train_size # 20% des données pour la validation
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True) # shuffle: on s'assure qu'il n'y ait pas un ordre "caché" que le modèle pourrait apprendre
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ====
# # Training setup
# ====
model = ResNet50Head(input_dim=X.shape[1]).to(DEVICE) # shape[1] sur tableau numpy = nombre de colonnes
criterion = nn.CrossEntropyLoss() # mesure à quel point le modèle se trompe
optimizer = torch.optim.Adam(model.parameters(), lr=LR) # ajuste les poids du modèle pour réduire l'erreur

# ====
# # MLflow Run
# ====
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run():
    # Log des hyperparamètres
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED,
        "device": DEVICE
    })
    
    print(f"\nTraining on {DEVICE}.\n")
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss_sum = 0.0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            
            train_loss_sum += loss.item()
        
        # Validation
        model.eval()
        y_true, y_pred = [], []
        
        with torch.no_grad(): # no_grad() = inférence seule, pas d'entrainement
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                preds = model(xb).argmax(1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(yb.numpy())
        
        # Calcul des métriques
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        avg_train_loss = train_loss_sum / len(train_loader)
        
        # Sauvegarde MLflow par epoch
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("val_acc", acc, step=epoch)
        mlflow.log_metric("val_f1", f1, step=epoch)
        mlflow.log_metric("val_precision", prec, step=epoch)
        mlflow.log_metric("val_recall", rec, step=epoch)
        
        print(f"Epoch {epoch+1:02d} | Loss={avg_train_loss:.4f} | Acc={acc:.4f}")
    
    # ====
    # Final report (weak val)
    # ====
    print("\n" + "=" * 50)
    print("Classification report (weak labels) [val split]:")
    print("=" * 50)
    report_weak = classification_report(y_true, y_pred, target_names=["Normal", "Cancer"])
    print(report_weak)
    mlflow.log_text(report_weak, "reports/weak_labels_report.txt")
    
    # ====
    # TEST DE VÉRITÉ: Évaluation sur le TEST MIXTE COMMUN
    # ====
    print("\n" + "=" * 50)
    print("TEST DE VÉRITÉ: Évaluation sur le TEST MIXTE COMMUN")
    print("=" * 50)
    
    model.eval()
    # Utilisation du test mixte (20 strong + 180 weak)
    X_test_mixed_tensor = torch.tensor(X_test_mixed, dtype=torch.float32).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(X_test_mixed_tensor)
        y_pred_mixte = outputs.argmax(1).cpu().numpy()
    
    # Métriques Mixtes (TEST COMMUN)
    acc_m = accuracy_score(y_test_mixed, y_pred_mixte)
    f1_m = f1_score(y_test_mixed, y_pred_mixte)
    prec_m = precision_score(y_test_mixed, y_pred_mixte, zero_division=0)
    rec_m = recall_score(y_test_mixed, y_pred_mixte, zero_division=0)
    
    mlflow.log_metrics({
        "test_acc": acc_m,
        "test_f1": f1_m,
        "test_precision": prec_m,
        "test_recall": rec_m
    })
    
    report_test = classification_report(
        y_test_mixed,
        y_pred_mixte,
        target_names=["Normal", "Cancer"]
    )
    
    print(report_test)
    mlflow.log_text(report_test, "reports/mixt_test_report.txt")
    
    # Log du modèle dans MLflow
    mlflow.pytorch.log_model(model, "model")


Chargement des features.

Samples weak-labeled: 1406
X shape: (1406, 2048)
Distribution: [ 354 1052]


2026/01/19 10:08:48 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet




Training on cpu.

Epoch 01 | Loss=0.3893 | Acc=0.9715
Epoch 02 | Loss=0.1078 | Acc=0.9593
Epoch 03 | Loss=0.0948 | Acc=0.9878
Epoch 04 | Loss=0.1030 | Acc=0.9553
Epoch 05 | Loss=0.0698 | Acc=0.9878
Epoch 06 | Loss=0.0468 | Acc=0.9837
Epoch 07 | Loss=0.0707 | Acc=0.9878
Epoch 08 | Loss=0.0372 | Acc=0.9919
Epoch 09 | Loss=0.0414 | Acc=0.9878
Epoch 10 | Loss=0.0446 | Acc=0.9919
Epoch 11 | Loss=0.0499 | Acc=0.9878
Epoch 12 | Loss=0.0429 | Acc=0.9919
Epoch 13 | Loss=0.0261 | Acc=0.9919
Epoch 14 | Loss=0.0382 | Acc=0.9878
Epoch 15 | Loss=0.0534 | Acc=0.9878
Epoch 16 | Loss=0.0230 | Acc=0.9878
Epoch 17 | Loss=0.0348 | Acc=0.9837
Epoch 18 | Loss=0.0262 | Acc=0.9878
Epoch 19 | Loss=0.0354 | Acc=0.9878
Epoch 20 | Loss=0.0232 | Acc=0.9919
Epoch 21 | Loss=0.0246 | Acc=0.9878
Epoch 22 | Loss=0.0127 | Acc=0.9919
Epoch 23 | Loss=0.0303 | Acc=0.9919
Epoch 24 | Loss=0.0413 | Acc=0.9837
Epoch 25 | Loss=0.0268 | Acc=0.9878
Epoch 26 | Loss=0.0189 | Acc=0.9878
Epoch 27 | Loss=0.0248 | Acc=0.9878
Epoch 28 

2026/01/19 10:08:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 30 | Loss=0.0225 | Acc=0.9878

Classification report (weak labels) [val split]:
              precision    recall  f1-score   support

      Normal       0.98      0.97      0.98        64
      Cancer       0.99      0.99      0.99       182

    accuracy                           0.99       246
   macro avg       0.99      0.98      0.98       246
weighted avg       0.99      0.99      0.99       246


TEST DE VÉRITÉ: Évaluation sur le TEST MIXTE COMMUN
              precision    recall  f1-score   support

      Normal       1.00      0.96      0.98        55
      Cancer       0.99      1.00      0.99       145

    accuracy                           0.99       200
   macro avg       0.99      0.98      0.99       200
weighted avg       0.99      0.99      0.99       200

🏃 View run aged-yak-878 at: http://mlflow:5000/#/experiments/136364928939801491/runs/f78ddc72455a492db711a50d13ccf4e2
🧪 View experiment at: http://mlflow:5000/#/experiments/136364928939801491


# Approche supervisée

In [3]:
# ====
# MLflow Config (Supervised)
# ====
MLFLOW_EXPERIMENT_NAME_SUP = "resnet50-supervised"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME_SUP)

with mlflow.start_run():
    # Log des hyperparamètres + infos split
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED,
        "device": DEVICE,
        "split": "train_test_split",
        "train_ratio": 0.8,
        "test_ratio": 0.2,
        "stratify": True,
        "input_dim": int(X_train_s.shape[1]),
        "n_train": int(len(y_train_s)),
        "n_test": int(len(y_test_mixed)),
    })
    
    # Créer les loaders
    train_loader_s = DataLoader(
        FeatureDataset(X_train_s, y_train_s),
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    test_tensor_mixed = torch.tensor(X_test_mixed, dtype=torch.float32).to(DEVICE)
    
    # Modèle ResNet50
    model_baseline = ResNet50Head(input_dim=X_train_s.shape[1]).to(DEVICE)
    optimizer_b = torch.optim.Adam(model_baseline.parameters(), lr=LR)
    
    # Entraînement sur les 80 images strong
    for epoch in range(EPOCHS):
        model_baseline.train()
        train_loss_sum = 0.0
        train_n_batches = 0
        
        for xb, yb in train_loader_s:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            
            optimizer_b.zero_grad()
            preds = model_baseline(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer_b.step()
            
            train_loss_sum += loss.item()
            train_n_batches += 1
        
        avg_train_loss = train_loss_sum / max(train_n_batches, 1)
        
        # Évaluation sur le test mixte à chaque epoch
        model_baseline.eval()
        with torch.no_grad():
            y_pred_epoch = model_baseline(test_tensor_mixed).argmax(1).cpu().numpy()
        
        acc = accuracy_score(y_test_mixed, y_pred_epoch)
        f1 = f1_score(y_test_mixed, y_pred_epoch)
        prec = precision_score(y_test_mixed, y_pred_epoch, zero_division=0)
        rec = recall_score(y_test_mixed, y_pred_epoch, zero_division=0)
        
        # Log MLflow par epoch
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("test_acc", acc, step=epoch)
        mlflow.log_metric("test_f1", f1, step=epoch)
        mlflow.log_metric("test_precision", prec, step=epoch)
        mlflow.log_metric("test_recall", rec, step=epoch)
        
        print(f"Epoch {epoch+1:02d} | Loss={avg_train_loss:.4f} | Acc={acc:.4f}")
    
    # Évaluation finale sur le test mixte
    model_baseline.eval()
    with torch.no_grad():
        y_pred_baseline = model_baseline(test_tensor_mixed).argmax(1).cpu().numpy()
    
    print("\nRapport Baseline sur test mixte (20 strong + 180 weak):")
    report_sup = classification_report(
        y_test_mixed, y_pred_baseline,
        target_names=['Normal', 'Cancer'],
        digits=4
    )
    print(report_sup)
    
    # Log artifact: report texte
    mlflow.log_text(report_sup, "reports/supervised_mixed_test_report.txt")
    
    # Log métriques finales
    mlflow.log_metrics({
        "final_test_acc": accuracy_score(y_test_mixed, y_pred_baseline),
        "final_test_f1": f1_score(y_test_mixed, y_pred_baseline),
        "final_test_precision": precision_score(y_test_mixed, y_pred_baseline, zero_division=0),
        "final_test_recall": recall_score(y_test_mixed, y_pred_baseline, zero_division=0)
    })
    
    # Sauvegarde du modèle
    save_path_sup = FEATURES_DIR / "resnet50_head_supervised_trained.pth"
    torch.save(model_baseline.state_dict(), save_path_sup)
    print(f"\nModèle supervisé sauvegardé: {save_path_sup}")
    
    mlflow.log_artifact(str(save_path_sup), "weights")
    mlflow.pytorch.log_model(model_baseline, "model")

Epoch 01 | Loss=0.6896 | Acc=0.9150
Epoch 02 | Loss=0.5911 | Acc=0.7200
Epoch 03 | Loss=0.6243 | Acc=0.9500
Epoch 04 | Loss=0.4579 | Acc=0.8550
Epoch 05 | Loss=0.3884 | Acc=0.9500
Epoch 06 | Loss=0.3383 | Acc=0.9400
Epoch 07 | Loss=0.2271 | Acc=0.9700
Epoch 08 | Loss=0.2862 | Acc=0.9450
Epoch 09 | Loss=0.1653 | Acc=0.9700
Epoch 10 | Loss=0.1459 | Acc=0.9650
Epoch 11 | Loss=0.1198 | Acc=0.9650
Epoch 12 | Loss=0.1172 | Acc=0.9700
Epoch 13 | Loss=0.1334 | Acc=0.9700
Epoch 14 | Loss=0.1815 | Acc=0.9650
Epoch 15 | Loss=0.0994 | Acc=0.9650
Epoch 16 | Loss=0.1055 | Acc=0.9650
Epoch 17 | Loss=0.0778 | Acc=0.9600
Epoch 18 | Loss=0.0499 | Acc=0.9650
Epoch 19 | Loss=0.0562 | Acc=0.9650
Epoch 20 | Loss=0.0763 | Acc=0.9600
Epoch 21 | Loss=0.0843 | Acc=0.9550
Epoch 22 | Loss=0.1727 | Acc=0.9650
Epoch 23 | Loss=0.0880 | Acc=0.9550
Epoch 24 | Loss=0.0492 | Acc=0.9300
Epoch 25 | Loss=0.1114 | Acc=0.9550
Epoch 26 | Loss=0.0240 | Acc=0.9600
Epoch 27 | Loss=0.0357 | Acc=0.9250
Epoch 28 | Loss=0.0570 | Acc

2026/01/19 10:09:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 30 | Loss=0.0248 | Acc=0.9500

Rapport Baseline sur test mixte (20 strong + 180 weak):
              precision    recall  f1-score   support

      Normal     0.9592    0.8545    0.9038        55
      Cancer     0.9470    0.9862    0.9662       145

    accuracy                         0.9500       200
   macro avg     0.9531    0.9204    0.9350       200
weighted avg     0.9504    0.9500    0.9491       200


Modèle supervisé sauvegardé: ../.data/features/resnet50_head_supervised_trained.pth
🏃 View run kindly-sloth-917 at: http://mlflow:5000/#/experiments/881879557287879727/runs/4d99d85914ed4833a9291d14b6c05401
🧪 View experiment at: http://mlflow:5000/#/experiments/881879557287879727
